<a href="https://colab.research.google.com/github/strongman2026/colab/blob/main/comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ComfyUI Colab Notebook

**运行顺序（首次）：** Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5 → Cell 6 → Cell 7 → Cell 8  
**日常启动（环境已就绪）：** 直接运行 Cell 8

| Cell | 功能 |
|------|------|
| 1 | 配置常量与工具函数 |
| 2 | 读取 Secrets（HF / Kaggle / FRP） |
| 3 | 部署 ComfyUI、依赖与 FRP 客户端 |
| 4 | 下载模型（HuggingFace） |
| 5 | 下载模型（Kaggle 数据集，可选） |
| 6 | 建立模型软链接到 ComfyUI 目录 |
| 7 | 安装自定义节点 |
| 8 | 启动 ComfyUI + FRP 隧道 |

> 所有 Cell 均支持幂等重复运行，已存在的文件/链接不会重复创建。

In [ ]:
# Cell 1 — 配置常量与工具函数
import os, sys, subprocess, time, socket, shutil
from pathlib import Path

# ── 运行环境 ────────────────────────────────────────────────────────
# 切换为 "kaggle" 可使用 Kaggle 专属路径
RUNTIME = "colab"  # "colab" | "kaggle"

# ── 路径配置 ────────────────────────────────────────────────────────
BASE_DIR    = Path("/content")         if RUNTIME == "colab" else Path("/kaggle/working")
COMFY_DIR   = BASE_DIR / "ComfyUI"
FRP_DIR     = BASE_DIR / "frp"
DATASET_DIR = BASE_DIR / "dataset"
FRPC_BIN    = FRP_DIR / "frpc"
MODELS_DIR  = COMFY_DIR / "models"

# ── VPS / FRP 配置 ──────────────────────────────────────────────────
VPS_IP       = "34.153.197.18"
FRP_PORT     = 7000
COMFYUI_PORT = 8188
REMOTE_PORT  = 8188

# ── 工具函数 ─────────────────────────────────────────────────────────
def run(cmd, check=True):
    """执行 shell 命令，失败时抛出异常。"""
    return subprocess.run(cmd, shell=True, check=check)


def ensure_dir(*paths):
    """确保目录存在，不存在则递归创建。"""
    for p in paths:
        Path(p).mkdir(parents=True, exist_ok=True)


def safe_remove(path):
    """安全删除文件、软链接或目录。"""
    p = Path(path)
    if p.is_symlink() or p.is_file():
        p.unlink()
    elif p.is_dir():
        shutil.rmtree(p)


def safe_symlink(src, dst, desc=""):
    """幂等软链接：目标已存在则先删除再创建；源文件不存在则给出提示。"""
    src, dst = Path(src), Path(dst)
    if not src.exists():
        print(f"⚠️  {desc}: 源文件不存在，跳过 ({src})")
        return
    ensure_dir(dst.parent)
    if dst.exists() or dst.is_symlink():
        safe_remove(dst)
    dst.symlink_to(src)
    print(f"✅ {desc}: {src} → {dst}")


def install_requirements(req_path, desc=""):
    """若 requirements.txt 存在则安装依赖，否则跳过。"""
    req = Path(req_path)
    if req.exists():
        run(f'{sys.executable} -m pip install -q -r "{req}"')
        print(f"✅ 已安装依赖: {desc}")
    else:
        print(f"⏭️  跳过依赖安装 (文件不存在): {req}")


print("✅ Cell 1 完成：配置与工具函数加载成功")


In [ ]:
# Cell 2 — 读取 Secrets（HuggingFace / Kaggle / FRP）
# 请在左侧 🔑 Secrets 面板中添加以下密钥并允许 Notebook 访问：
#   HF_TOKEN         — HuggingFace 访问令牌
#   KAGGLE_API_TOKEN — Kaggle API 令牌
#   FRP_TOKEN        — FRP 服务端鉴权密钥

if RUNTIME == "colab":
    from google.colab import userdata

    def _load_secret(key, env_key=None):
        env_key = env_key or key
        try:
            os.environ[env_key] = userdata.get(key)
            print(f"✅ 已读取 {key}")
        except Exception:
            print(f"⚠️  未找到 {key}，请在左侧 🔑 Secrets 中添加")

    _load_secret("HF_TOKEN")
    _load_secret("KAGGLE_API_TOKEN")
    _load_secret("FRP_TOKEN")

elif RUNTIME == "kaggle":
    # Kaggle 环境中，KAGGLE_* 变量由平台自动注入，无需手动设置
    print("ℹ️  Kaggle 环境：KAGGLE_* 变量已由平台自动配置")
    if not os.environ.get("FRP_TOKEN"):
        print("⚠️  FRP_TOKEN 未设置，FRP 隧道鉴权将失败")

print("✅ Cell 2 完成：Secrets 读取完毕")


In [ ]:
# Cell 3 — 部署 ComfyUI、依赖与 FRP 客户端（幂等）

# ── Google Drive（仅 Colab）──────────────────────────────────────────
if RUNTIME == "colab":
    from google.colab import drive
    print("☁️  挂载 Google Drive (弹窗请点击允许)...")
    drive.mount("/content/drive")

# ── ComfyUI ──────────────────────────────────────────────────────────
if not COMFY_DIR.exists():
    print("📥 克隆 ComfyUI...")
    run(f"git clone -q https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}")
    run(f"git clone -q https://github.com/ltdrdata/ComfyUI-Manager.git "
        f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager")
else:
    print("⏭️  ComfyUI 已存在，跳过克隆")

install_requirements(COMFY_DIR / "requirements.txt", "ComfyUI")

# ── FRP 客户端 ────────────────────────────────────────────────────────
if not FRPC_BIN.exists():
    print("🔧 部署 FRP 客户端...")
    ensure_dir(FRP_DIR)
    run(f"wget -q -O {FRP_DIR}/frp.tar.gz "
        "https://github.com/fatedier/frp/releases/download/v0.61.1/"
        "frp_0.61.1_linux_amd64.tar.gz")
    run(f"tar -xzf {FRP_DIR}/frp.tar.gz -C {FRP_DIR} --strip-components=1")
    run(f"chmod +x {FRPC_BIN}")
else:
    print("⏭️  FRP 客户端已存在，跳过部署")

# ── FRP 配置文件（使用 Secrets 中的 FRP_TOKEN）────────────────────────
frp_token = os.environ.get("FRP_TOKEN", "")
frpc_config = f"""serverAddr = \"{VPS_IP}\"
serverPort = {FRP_PORT}
auth.method = \"token\"
auth.token = \"{frp_token}\"

[[proxies]]
name = \"comfyui_colab\"
type = \"tcp\"
localIP = \"127.0.0.1\"
localPort = {COMFYUI_PORT}
remotePort = {REMOTE_PORT}
"""
(FRP_DIR / "frpc.toml").write_text(frpc_config, encoding="utf-8")

print("✅ Cell 3 完成：基础环境部署成功")


In [ ]:
# Cell 4 — 下载模型（HuggingFace）
# 需要先运行 Cell 2 以确保 HF_TOKEN 已设置

run("apt-get install -y -qq aria2")
ensure_dir(DATASET_DIR, MODELS_DIR / "vae", MODELS_DIR / "clip")

def _hf_download(url, dest_dir, filename, desc):
    """使用 aria2c 从 HuggingFace 下载文件，已存在则跳过。"""
    dest = Path(dest_dir) / filename
    if dest.exists():
        print(f"⏭️  已存在，跳过下载: {desc}")
        return
    print(f"📥 下载 {desc}...")
    run(f'aria2c --header="Authorization: Bearer $HF_TOKEN" '
        f'--console-log-level=error -c -x 16 -s 16 -k 1M '
        f'"{url}" -d "{dest_dir}" -o "{filename}"')

# Flux 主模型 — VAE
_hf_download(
    "https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors",
    DATASET_DIR, "ae.safetensors", "VAE (ae.safetensors)"
)

# 文本编码器 — CLIP-L
_hf_download(
    "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors",
    DATASET_DIR, "clip_l.safetensors", "CLIP-L (clip_l.safetensors)"
)

# 文本编码器 — T5XXL
_hf_download(
    "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors",
    DATASET_DIR, "t5xxl_fp16.safetensors", "T5XXL FP16 (t5xxl_fp16.safetensors, ~9.8 GB)"
)

print("✅ Cell 4 完成：HuggingFace 模型下载完毕")


In [ ]:
# Cell 5 — 下载模型（Kaggle 数据集，可选）
# 需要先运行 Cell 2 以确保 KAGGLE_API_TOKEN 已设置

run("pip install -q -U kaggle")
ensure_dir(DATASET_DIR)

def _kaggle_download(dataset, dest_dir, desc):
    """从 Kaggle 下载数据集并解压，目录非空则跳过。"""
    dest = Path(dest_dir)
    if dest.exists() and any(dest.iterdir()):
        print(f"⏭️  目标目录非空，跳过下载: {desc}")
        return
    print(f"📥 正在从 Kaggle 下载: {desc}...")
    run(f'kaggle datasets download -d "{dataset}" -p "{dest_dir}" --unzip')

# Flux 主模型套装（含 flux1-schnell.safetensors）
_kaggle_download(
    "jarredstroman/flux-2d-assets-2026-v12",
    DATASET_DIR,
    "Flux 2D Assets 2026 v12"
)

print("✅ Cell 5 完成：Kaggle 模型下载完毕")


In [ ]:
# Cell 6 — 建立模型软链接到 ComfyUI 目录（幂等）

ensure_dir(
    MODELS_DIR / "unet",
    MODELS_DIR / "checkpoints",
    MODELS_DIR / "clip",
    MODELS_DIR / "vae",
    MODELS_DIR / "ipadapter",
    MODELS_DIR / "clip_vision",
    MODELS_DIR / "controlnet",
    COMFY_DIR / "output",
)

# ── Flux 主模型 ───────────────────────────────────────────────────────
# Flux 使用 UNETLoader，主模型放 unet/ 目录
safe_symlink(
    DATASET_DIR / "flux1-schnell.safetensors",
    MODELS_DIR / "unet" / "flux1-schnell.safetensors",
    "Flux 主模型"
)

# ── VAE ───────────────────────────────────────────────────────────────
safe_symlink(
    DATASET_DIR / "ae.safetensors",
    MODELS_DIR / "vae" / "ae.safetensors",
    "VAE (ae.safetensors)"
)

# ── 文本编码器 ────────────────────────────────────────────────────────
safe_symlink(
    DATASET_DIR / "clip_l.safetensors",
    MODELS_DIR / "clip" / "clip_l.safetensors",
    "CLIP-L"
)
safe_symlink(
    DATASET_DIR / "t5xxl_fp16.safetensors",
    MODELS_DIR / "clip" / "t5xxl_fp16.safetensors",
    "T5XXL FP16"
)

# ── IPAdapter 权重（可选，下载后取消注释）─────────────────────────────
# safe_symlink(
#     DATASET_DIR / "flux-ip-adapter.safetensors",
#     MODELS_DIR / "ipadapter" / "flux-ip-adapter.safetensors",
#     "Flux IPAdapter"
# )

# ── CLIP Vision（可选，IPAdapter 依赖）───────────────────────────────
# safe_symlink(
#     DATASET_DIR / "CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors",
#     MODELS_DIR / "clip_vision" / "CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors",
#     "CLIP Vision"
# )

# ── ControlNet（可选）────────────────────────────────────────────────
# safe_symlink(
#     DATASET_DIR / "controlnet_canny.safetensors",
#     MODELS_DIR / "controlnet" / "controlnet_canny.safetensors",
#     "ControlNet Canny"
# )

print("\n📁 当前 models 目录结构：")
run(f"find {MODELS_DIR} -maxdepth 2 | sort", check=False)

print("✅ Cell 6 完成：模型软链接建立完毕")


In [ ]:
# Cell 7 — 安装自定义节点（幂等）

CUSTOM_NODES_DIR = COMFY_DIR / "custom_nodes"
ensure_dir(CUSTOM_NODES_DIR)

# 格式: (目录名, Git 仓库 URL)
CUSTOM_NODES = [
    ("ComfyUI_IPAdapter_plus",   "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git"),
    ("comfyui_controlnet_aux",   "https://github.com/Fannovel16/comfyui_controlnet_aux.git"),
    ("rgthree-comfy",            "https://github.com/rgthree/rgthree-comfy.git"),
]

for name, repo in CUSTOM_NODES:
    target = CUSTOM_NODES_DIR / name
    if not target.exists():
        print(f"📥 克隆 {name}...")
        run(f"git clone -q {repo} {target}")
    else:
        print(f"⏭️  已存在，跳过克隆: {name}")
    install_requirements(target / "requirements.txt", name)

print("✅ Cell 7 完成：自定义节点安装完毕")


In [ ]:
# Cell 8 — 启动 ComfyUI + FRP 隧道
# 日常只需运行本 Cell（前提：Cell 1–7 已至少执行过一次）

# ── 应用商店开关 ──────────────────────────────────────────────────────
# False = 屏蔽 ComfyUI-Manager，加快启动速度
# True  = 启用 ComfyUI-Manager，可在线安装/更新插件
ENABLE_MANAGER = False

# ── 终止旧进程 ────────────────────────────────────────────────────────
run("pkill -f 'main.py'", check=False)
run("pkill -f frpc", check=False)
time.sleep(1)

# ── Manager 状态切换 ──────────────────────────────────────────────────
MANAGER_DIR        = COMFY_DIR / "custom_nodes" / "ComfyUI-Manager"
HIDDEN_MANAGER_DIR = COMFY_DIR / "ComfyUI-Manager_hidden"

if ENABLE_MANAGER:
    if HIDDEN_MANAGER_DIR.exists():
        HIDDEN_MANAGER_DIR.rename(MANAGER_DIR)
        print("✅ ComfyUI-Manager 已启用")
else:
    if MANAGER_DIR.exists():
        MANAGER_DIR.rename(HIDDEN_MANAGER_DIR)
        print("✅ ComfyUI-Manager 已隐藏（加速启动）")

# ── 启动 ComfyUI ──────────────────────────────────────────────────────
print("🚀 启动 ComfyUI...")
comfy = subprocess.Popen(
    [sys.executable, "main.py", "--normalvram"],
    cwd=str(COMFY_DIR)
)

# ── 等待 ComfyUI 就绪 ─────────────────────────────────────────────────
print(f"⏳ 等待 ComfyUI 在 {COMFYUI_PORT} 端口就绪（最多 60 秒）...")
for _ in range(60):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", COMFYUI_PORT)) == 0:
            break
    time.sleep(1)
else:
    raise RuntimeError(f"ComfyUI 启动失败：{COMFYUI_PORT} 端口在 60 秒内未就绪")

# ── 启动 FRP 隧道 ─────────────────────────────────────────────────────
print("🌐 启动 FRP 隧道...")
frpc = subprocess.Popen(
    [str(FRPC_BIN), "-c", str(FRP_DIR / "frpc.toml")]
)

print("\n" + "=" * 50)
print(f"✅ ComfyUI 已启动，请访问: http://{VPS_IP}:{REMOTE_PORT}")
print(f"   Manager: {'已加载' if ENABLE_MANAGER else '已隐藏'}")
print("=" * 50)
# 注意：不调用 comfy.wait()，保持 Cell 非阻塞
